# A simple livability index for Buenos Aires

Dimensions:
- Accessibility and mobility
- Green spaces and recreation
- Public services and amenities
- Safety and security
- Economic opportunities

## Step 1: data collection by district (_comuna_)

In [0]:
import os
import osmnx as ox
import pandas as pd

In [0]:
def download_profile_by_place(place, folderout):
    admin_district = ox.geocode_to_gdf(place)
    admin_district.to_parquet(folderout + "/admin_district.parquet")

    # Accesibility and mobility
    public_transport = ox.features_from_place(
        place_name, tags={"public_transport": True}
    )

    public_transport.to_parquet(folderout + "/public_transport.parquet")

    # Green spaces and recreation
    parks_and_recreation = ox.features_from_place(place, tags={"leisure": True})

    parks_and_recreation.to_parquet(folderout + "/parks_and_recreation.parquet")

    # Public services
    healthcare = ox.features_from_place(
        place, tags={"amenity": ["hospital", "pharmacy"]}
    )
    education = ox.features_from_place(
        place, tags={"amenity": ["kindergarten", "school", "university"]}
    )

    healthcare.to_parquet(folderout + "/healthcare.parquet")
    education.to_parquet(folderout + "/education.parquet")

    # Safety and security
    emergency_services = ox.features_from_place(
        place, tags={"amenity": ["police", "fire_station", "clinic"]}
    )

    emergency_services.to_parquet(folderout + "/emergency_services.parquet")

    # Economic opportunities
    commerce = ox.features_from_place("CABA", tags={"shop": True})
    employment = ox.features_from_place(
        "CABA", tags={"office": True, "industrial": True}
    )

    commerce.to_parquet(folderout + "/commerce.parquet")
    employment.to_parquet(folderout + "/employment.parquet")

In [0]:
for i in range(1, 16):
    place_name = f"Comuna {i:{2}}, CABA"

    # A location to save the data files to
    folderout = f"data/raw/comuna{i:02}"
    if not os.path.exists(folderout):
        os.makedirs(folderout)

    download_profile_by_place(place_name, folderout)

## Step 2: computing index by district

In [0]:
import os
import osmnx as ox
import pandas as pd
import geopandas as gpd

from sklearn.preprocessing import MinMaxScaler

CRS = "EPSG:9498"

### Deriving district metrics

In [0]:
def characterize_district(place, folderin, crs):
    admin_district = gpd.read_parquet(folderin + "/admin_district.parquet")
    public_transport = gpd.read_parquet(folderin + "/public_transport.parquet")
    parks_and_recreation = gpd.read_parquet(folderin + "/parks_and_recreation.parquet")
    healthcare = gpd.read_parquet(folderin + "/healthcare.parquet")
    education = gpd.read_parquet(folderin + "/education.parquet")
    emergency_services = gpd.read_parquet(folderin + "/emergency_services.parquet")
    commerce = gpd.read_parquet(folderin + "/commerce.parquet")
    employment = gpd.read_parquet(folderin + "/employment.parquet")

    district_area = (admin_district.to_crs(crs).geometry.area / 1000 ** 2)[0]

    data = {
        "place": place,
        "public_transport_density": len(public_transport) / district_area,
        "parks_and_recreation_density": len(parks_and_recreation) / district_area,
        "parks_and_recreation_coverage": parks_and_recreation.to_crs(crs).area.sum() / (district_area * 1000 ** 2),
        "healthcare_density": len(healthcare) / district_area,
        "education_density": len(education) / district_area,
        "emergency_services_density": len(emergency_services) / district_area,
        "commerce_density": len(commerce) / district_area,
        "employment_density": len(employment) / district_area,
    }

    return pd.DataFrame([data])

In [0]:
districts_profiles = []

for i in range(1, 16):
    place = f"Comuna {i}, CABA"
    folderin = f"data/raw/comuna{i:02}"

    districts_profiles.append(characterize_district(place, folderin, CRS))

districts_profiles = pd.concat(districts_profiles)

In [0]:
districts_profiles.describe()